# Занятие 1. Семинар: kNN и наивный Байес

Сегодня две задачи. Первая — убедиться, что у всех работает окружение.
Вторая — своими руками написать два первых алгоритма машинного обучения
и посмотреть, как те же самые вещи выглядят в библиотеке.

Ничего не скачиваем из интернета: все данные приезжают вместе с scikit-learn.

**Как работать с ноутбуком.** Ячейки идут по порядку, запускать сверху вниз.
Там, где написано `# TODO: ваш код`, мы пишем код вместе на паре.
Сразу за каждой такой ячейкой идет ячейка с проверками: если она отработала
без ошибок и напечатала, что все в порядке, можно идти дальше.
Решенная версия появится в репозитории вечером.

## 0. Окружение

Проверяем, что все встало. Если какая-то строчка ругается, зовите меня.

In [ ]:
import sys
import platform

print("python", platform.python_version())

for name in ["numpy", "pandas", "matplotlib", "sklearn"]:
    try:
        module = __import__(name)
        print(f"  ok       {name} {getattr(module, '__version__', '?')}")
    except ImportError:
        print(f"  MISSING  {name}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

SEED = 42
rng = np.random.default_rng(SEED)

## 1. Данные

Датасет про вино: химический анализ вин трех сортов, выращенных в одном
регионе Италии. По результатам анализа нужно определить сорт.

Он маленький и приезжает вместе с библиотекой, так что подходит,
чтобы разобраться в механике, не отвлекаясь на загрузку данных.

In [ ]:
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
data = wine.frame

print("объектов:", data.shape[0])
print("колонок: ", data.shape[1])
data.head()

Разберемся, что здесь что.

- `data` — таблица: строка это один объект, колонка это один признак;
- `target` — то, что мы предсказываем, номер сорта;
- остальные тринадцать колонок — то, из чего предсказываем

Слова, которые дальше будут повторяться весь семестр: **объект**, **признак**,
**целевая переменная**. Объект это строка, признак это колонка, целевая
переменная это отдельная колонка, ради которой все затевается.

In [ ]:
print("названия признаков:")
for i, name in enumerate(wine.feature_names):
    print(f"  {i:2d}  {name}")

print()
print("классы:", dict(zip(range(3), wine.target_names)))
print()
print(data["target"].value_counts().sort_index())

### Первое, на что всегда стоит посмотреть

Не на модель, а на масштаб признаков. Запомните этот вывод, через
двадцать минут он нам пригодится.

In [ ]:
summary = data.drop(columns="target").describe().T[["mean", "std", "min", "max"]]
summary.sort_values("max", ascending=False).head(6)

Разброс колоссальный. У `proline` значения около тысячи, у `nonflavanoid_phenols` —
десятые доли. Разница в четыре порядка. Пока просто отметим это.

## 2. Постановка задачи

Формально задача обучения с учителем выглядит так.

Есть таблица признаков $X$ размера $n \times d$: $n$ объектов, $d$ признаков.
Есть вектор ответов $y$ длины $n$.

Нужно построить **решающее правило** — функцию, которая по объекту
выдает ответ:

$$a: \mathcal{X} \rightarrow \mathcal{Y}$$

Здесь $\mathcal{X}$ — множество всех возможных объектов, $\mathcal{Y}$ —
множество допустимых ответов. Для наших вин объект это набор из тринадцати
чисел, а ответ это один из трех сортов, то есть $\mathcal{Y} = \{0, 1, 2\}$.

Обозначение $a(x)$ будет встречаться до конца курса. Подали объект $x$ —
получили ответ $a(x)$. Все методы, которые мы пройдем, отличаются только
тем, **как именно устроена** эта функция и **как ее подбирают**
по обучающей выборке.

Задача обучения — найти такую $a$, которая хорошо отвечает не на тех
объектах, что мы уже видели, а на новых.

Слово «учитель» означает ровно одно: нам заранее показали правильные ответы
на части данных. Все остальное в машинном обучении — вариации на эту тему.

In [ ]:
X = data.drop(columns="target").to_numpy()
y = data["target"].to_numpy()

print("X:", X.shape, X.dtype)
print("y:", y.shape, y.dtype)

## 3. Как честно измерить качество

Соблазн такой: обучить модель на всех данных, на них же проверить, получить
высокий результат и обрадоваться. Так делать нельзя, и вот почему.

Представьте модель, которая просто запомнила всю таблицу. На знакомых объектах
она отвечает идеально, на новых — как повезет. Проверка на тех же данных,
на которых учились, измеряет память, а не умение обобщать.

Поэтому данные делятся заранее. Одна часть для обучения, вторая откладывается
и не трогается до самого конца.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=SEED,
    stratify=y,      # сохранить доли классов в обеих частях
)

print("train:", X_train.shape[0], "объектов")
print("test: ", X_test.shape[0], "объектов")
print()
print("доли классов в train:", np.round(np.bincount(y_train) / len(y_train), 3))
print("доли классов в test: ", np.round(np.bincount(y_test) / len(y_test), 3))

Параметр `stratify=y` следит, чтобы доли классов в обеих частях совпадали.
Без него на маленькой выборке легко получить перекос, и тогда результат
на тесте будет говорить больше о неудачном делении, чем о модели.

## 4. Baseline

Прежде чем строить модель, надо понять, с чем сравнивать. Простейшая
стратегия: всегда отвечать самым частым классом. Она ничего не знает
о признаках, и любая осмысленная модель обязана ее обыграть.

Число, которое получится, — точка отсчета на весь остальной семинар.

In [ ]:
# TODO: обучить DummyClassifier со стратегией most_frequent,
# сохранить его предсказания в y_pred_dummy, а accuracy в baseline_acc
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

raise NotImplementedError

In [ ]:
# --- проверка ---

# 1. accuracy это доля, она обязана лежать между нулем и единицей
assert 0.0 <= baseline_acc <= 1.0, "accuracy вне отрезка [0, 1]"

# 2. модель отвечает всегда одним и тем же классом
assert len(np.unique(y_pred_dummy)) == 1, "константная модель должна давать один класс"

# 3. и это самый частый класс обучающей выборки
most_frequent = np.bincount(y_train).argmax()
assert y_pred_dummy[0] == most_frequent, "предсказан не самый частый класс трейна"

# 4. значит accuracy равна доле этого класса в тесте
expected = (y_test == most_frequent).mean()
assert np.isclose(baseline_acc, expected), f"ожидали {expected:.3f}, получили {baseline_acc:.3f}"

print(f"проверки пройдены, baseline = {baseline_acc:.3f}")

**Accuracy** — доля правильных ответов. Первая метрика курса, и самая
обманчивая: на несбалансированных данных она врет так, что на нее нельзя
смотреть вообще. К этому вернемся в конце семинара.

## 5. kNN своими руками

Идея метода ближайших соседей помещается в одну фразу: **похожие объекты
имеют похожие ответы**.

Алгоритм для нового объекта:

1. посчитать расстояние до всех объектов обучающей выборки;
2. взять $k$ ближайших;
3. вернуть класс, который среди них встретился чаще

Обучения здесь нет вообще. Модель просто хранит выборку, а вся работа
происходит в момент предсказания. Такие методы называют ленивыми.

В терминах прошлого раздела: решающее правило $a(x)$ здесь устроено как
голосование соседей. Формулы с настраиваемыми параметрами у него нет,
вся модель — это сама обучающая выборка.

### Шаг 1. Расстояния

Начнем с евклидова расстояния между двумя векторами:

$$\rho(a, b) = \sqrt{\sum_{j=1}^{d} (a_j - b_j)^2}$$

Нам нужны расстояния от каждого тестового объекта до каждого обучающего,
то есть матрица размера $m \times n$, где $m$ объектов в запросе
и $n$ в обучающей выборке.

Циклом это писать долго и медленно. Вспомним, как в numpy устроен
**broadcasting**: если у двух массивов размерности не совпадают, numpy
растягивает единичные оси до нужного размера.

```python
A.shape  ->  (m, d)
B.shape  ->  (n, d)
```

Если превратить их в `(m, 1, d)` и `(1, n, d)`, то при вычитании numpy
растянет обе единичные оси и вернет массив `(m, n, d)` — все попарные
разности сразу. Дальше остается возвести в квадрат, просуммировать
по последней оси и взять корень.

Добавить ось можно через `None` в индексации: `A[:, None, :]` дает `(m, 1, d)`.

In [ ]:
def pairwise_distances(A, B):
    # A: (m, d), B: (n, d)  ->  матрица евклидовых расстояний (m, n)
    # TODO: ваш код
    raise NotImplementedError

In [ ]:
# --- проверка ---

a = np.array([[0.0, 0.0], [3.0, 4.0]])
b = np.array([[0.0, 0.0], [0.0, 1.0], [6.0, 8.0]])
D = pairwise_distances(a, b)

# 1. форма результата: строк столько же, сколько объектов в первом массиве
assert D.shape == (2, 3), f"ожидали форму (2, 3), получили {D.shape}"

# 2. расстояние от точки до себя самой равно нулю
assert np.isclose(D[0, 0], 0.0), "расстояние от точки до себя не ноль"

# 3. классический треугольник: катеты 3 и 4, гипотенуза 5
assert np.isclose(D[1, 0], 5.0), f"ожидали 5.0, получили {D[1, 0]}"

# 4. расстояния неотрицательны и симметричны
assert (D >= 0).all(), "появилось отрицательное расстояние"
assert np.allclose(pairwise_distances(b, a), D.T), "нарушена симметрия"

# 5. сверка с реализацией scikit-learn на настоящих данных
from sklearn.metrics.pairwise import euclidean_distances
assert np.allclose(pairwise_distances(X_test, X_train),
                   euclidean_distances(X_test, X_train)), "расходится со sklearn"

print("проверки пройдены")
print(D.round(3))

### Шаг 2. Голосование соседей

Теперь собираем метод целиком. Для каждого объекта запроса берем индексы
$k$ ближайших, смотрим их метки и возвращаем ту, что встретилась чаще.

`np.argsort` вернет индексы в порядке возрастания расстояния, первые $k$
из них и есть соседи.

In [ ]:
def knn_predict(X_train, y_train, X_query, k=5):
    # 1. расстояния от каждого объекта X_query до каждого объекта X_train
    # 2. индексы k ближайших соседей
    # 3. их метки
    # 4. самый частый класс среди соседей
    # TODO: ваш код
    raise NotImplementedError


y_pred_manual = knn_predict(X_train, y_train, X_test, k=5)
acc_manual = accuracy_score(y_test, y_pred_manual)
print(f"наш kNN, k=5: {acc_manual:.3f}")

In [ ]:
# --- проверка ---

# 1. на каждый объект запроса ровно одно предсказание
assert y_pred_manual.shape == y_test.shape, "не совпало число предсказаний"

# 2. предсказанные метки существуют среди классов трейна
assert set(np.unique(y_pred_manual)) <= set(np.unique(y_train)), "предсказан несуществующий класс"

# 3. при k=1 объект из обучающей выборки обязан предсказать сам себя
self_pred = knn_predict(X_train, y_train, X_train, k=1)
assert (self_pred == y_train).all(), "при k=1 объект должен попадать сам в себя"

# 4. при k равном размеру трейна ответ вырождается в самый частый класс
all_neighbors = knn_predict(X_train, y_train, X_test, k=len(X_train))
assert len(np.unique(all_neighbors)) == 1, "при k=n ответ должен быть константой"
assert all_neighbors[0] == np.bincount(y_train).argmax(), "константа не совпала с самым частым классом"

# 5. модель осмысленнее случайного угадывания
assert acc_manual > baseline_acc, "kNN не обыграл константный baseline"

print(f"проверки пройдены, accuracy = {acc_manual:.3f}")

## 6. То же самое в scikit-learn

Теперь сверимся с библиотекой. Если наш код правильный, результаты совпадут
до последнего объекта.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred_sklearn = knn.predict(X_test)

print(f"sklearn kNN, k=5: {accuracy_score(y_test, y_pred_sklearn):.3f}")
print("совпадает с нашим предсказанием:",
      np.array_equal(y_pred_manual, y_pred_sklearn))

Обратите внимание на интерфейс: `fit` учит, `predict` предсказывает.
Так устроены **все** модели в scikit-learn без исключения. Дальше в курсе
будут появляться десятки разных алгоритмов, и у каждого будут эти два метода.
Это позволяет менять модель одной строкой.

## 7. Ловушка масштаба

Вернемся к тому, что заметили в начале. У `proline` значения около тысячи,
у остальных признаков — единицы и доли.

Посмотрите на формулу расстояния еще раз. Разность по `proline` в тысячу
единиц дает вклад в миллион, разность по `hue` в половину единицы — вклад
в четверть. То есть расстояние определяется одним признаком, а остальные
двенадцать не участвуют вообще.

Лечится это стандартизацией: у каждого признака вычитается среднее
и результат делится на стандартное отклонение.

$$z_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$$

После такого преобразования у всех признаков среднее ноль и разброс единица,
и в расстояние они входят на равных.

In [ ]:
from sklearn.preprocessing import StandardScaler

# TODO: получить X_train_scaled и X_test_scaled, обучить на них kNN
# и сравнить accuracy с моделью на исходных данных.
# Результаты положить в acc_raw и acc_scaled
raise NotImplementedError

In [ ]:
# --- проверка ---

# 1. после стандартизации у обучающей части среднее ноль, разброс единица
assert np.allclose(X_train_scaled.mean(axis=0), 0, atol=1e-9), "среднее трейна не ноль"
assert np.allclose(X_train_scaled.std(axis=0), 1, atol=1e-9), "разброс трейна не единица"

# 2. у тестовой части среднее НЕ обязано быть нулем: параметры взяты с трейна
assert not np.allclose(X_test_scaled.mean(axis=0), 0, atol=1e-6), \
    "похоже, scaler обучен на тесте, а так делать нельзя"

# 3. форма данных не изменилась
assert X_train_scaled.shape == X_train.shape and X_test_scaled.shape == X_test.shape

# 4. масштабирование улучшило качество, причем заметно
assert acc_scaled > acc_raw, "после масштабирования качество не выросло"
print(f"проверки пройдены, прирост {acc_scaled - acc_raw:+.3f}")

Разница огромная. Один вызов `StandardScaler` дал больше, чем дал бы любой
подбор гиперпараметров.

Отдельно обратите внимание на две строчки:

```python
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
```

На обучающей части — `fit_transform`, на тестовой — только `transform`.
Среднее и стандартное отклонение считаются **исключительно по трейну**.
Если посчитать их по всем данным сразу, информация о тесте просочится
в обучение. Это называется утечкой, и это одна из самых частых ошибок
в машинном обучении. Отдельное занятие про это будет четвертым.

Третья проверка в ячейке выше ловит ровно эту ошибку: если бы вы обучили
`scaler` на тесте, среднее теста оказалось бы нулевым и проверка упала бы.

### Насколько сильно масштаб перекосил расстояния

Посмотрим на вклад каждого признака в суммарное расстояние.

In [ ]:
variance = X_train.var(axis=0)
share = variance / variance.sum()

order = np.argsort(share)[::-1]
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(range(len(share)), share[order] * 100, color="#0072CE")
ax.set_xticks(range(len(share)))
ax.set_xticklabels([wine.feature_names[i] for i in order], rotation=60, ha="right")
ax.set_ylabel("вклад в расстояние, %")
ax.set_title("До масштабирования один признак решает все")
plt.tight_layout()
plt.show()

print(f"доля proline: {share[order][0] * 100:.2f}%")

## 8. Выбор k

Единственный параметр метода. Что происходит на краях:

- `k = 1` — ответ определяется одним ближайшим объектом. Модель подстраивается
  под каждую точку, включая шум. Это переобучение;
- `k = n` — соседями оказывается вся выборка, и ответ всегда один и тот же,
  самый частый класс. Это наш baseline, то есть недообучение.

Истина где-то посередине, и найти ее можно только экспериментом.

In [ ]:
k_values = list(range(1, 41))
train_scores, test_scores = [], []

# TODO: для каждого k обучить kNN на отмасштабированных данных,
# записать accuracy на train в train_scores и на test в test_scores,
# построить два графика на одних осях, найти best_k
raise NotImplementedError

In [ ]:
# --- проверка ---

# 1. по одному значению на каждое k
assert len(train_scores) == len(k_values) == len(test_scores), "не совпала длина списков"

# 2. все значения это доли
assert all(0.0 <= s <= 1.0 for s in train_scores + test_scores), "accuracy вне [0, 1]"

# 3. при k=1 модель отвечает идеально на собственной обучающей выборке
assert np.isclose(train_scores[0], 1.0), "при k=1 train accuracy обязана быть равна 1"

# 4. на тесте при k=1 такого нет: там модель уже ошибается
assert test_scores[0] < 1.0, "на тесте при k=1 не должно быть идеального результата"

# 5. лучшее k выбрано корректно
assert test_scores[k_values.index(best_k)] == max(test_scores), "best_k не соответствует максимуму"

print(f"проверки пройдены: train при k=1 равен {train_scores[0]:.3f}, "
      f"test {test_scores[0]:.3f}")

На графике видно главное, ради чего он строился: при `k = 1` качество на трейне
равно единице. Модель отвечает идеально на данных, которые видела, потому что
ближайший сосед объекта — он сам. На тесте при этом качество ниже.

Расхождение между двумя кривыми и есть переобучение, увиденное глазами.

**Важная оговорка про честность.** Мы только что выбрали `k` по тестовой
выборке. Строго говоря, так делать нельзя: тест перестает быть независимой
проверкой, если по нему что-то подбирали. Правильный способ — отдельная
валидационная часть или кросс-валидация. Разберем на четвертом занятии,
пока просто зафиксируйте, что здесь мы схитрили.

## 9. Метрики: почему accuracy недостаточно

Возьмем другой датасет — диагностика опухоли молочной железы, два класса.
И построим модель, которая всегда отвечает «доброкачественная».

In [ ]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    cancer.data, cancer.target, test_size=0.3, random_state=SEED, stratify=cancer.target
)

print("классы:", dict(zip(range(2), cancer.target_names)))
print("доли в тесте:", np.round(np.bincount(yc_test) / len(yc_test), 3))

lazy = DummyClassifier(strategy="most_frequent").fit(Xc_train, yc_train)
print(f"\naccuracy модели, которая всегда говорит одно и то же: "
      f"{accuracy_score(yc_test, lazy.predict(Xc_test)):.3f}")

Шестьдесят три процента правильных ответов у модели, которая не смотрит
на данные вообще. И это еще мягкий случай: при соотношении классов один
к тысяче такая модель наберет 99.9 процента.

Значит accuracy сама по себе ничего не говорит. Нужно смотреть, **какие
именно** ошибки делает модель. Для этого предсказания раскладывают
на четыре клетки.

|  | предсказано 1 | предсказано 0 |
|:--|:--|:--|
| **на самом деле 1** | TP, верно найденные | FN, пропущенные |
| **на самом деле 0** | FP, ложная тревога | TN, верно отвергнутые |

Отсюда растут две метрики:

$$\text{precision} = \frac{TP}{TP + FP}, \qquad
  \text{recall} = \frac{TP}{TP + FN}$$

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

# TODO: отмасштабировать данные (Xc_train_s, Xc_test_s), обучить kNN,
# получить предсказания yc_pred, построить матрицу ошибок
# и напечатать classification_report
raise NotImplementedError

In [ ]:
# --- проверка ---
from sklearn.metrics import confusion_matrix, precision_score, recall_score

cm = confusion_matrix(yc_test, yc_pred)

# 1. матрица два на два, потому что классов два
assert cm.shape == (2, 2), f"ожидали матрицу (2, 2), получили {cm.shape}"

# 2. сумма всех клеток равна числу объектов теста
assert cm.sum() == len(yc_test), "в матрицу попали не все объекты"

# 3. диагональ это правильные ответы, и их доля равна accuracy
assert np.isclose(np.trace(cm) / cm.sum(), accuracy_score(yc_test, yc_pred)), \
    "диагональ матрицы не сходится с accuracy"

# 4. считаем precision и recall руками и сверяем с библиотекой
tn, fp, fn, tp = cm.ravel()
assert np.isclose(tp / (tp + fp), precision_score(yc_test, yc_pred)), "precision не сошелся"
assert np.isclose(tp / (tp + fn), recall_score(yc_test, yc_pred)), "recall не сошелся"

print(f"проверки пройдены")
print(f"  TP={tp}  FP={fp}")
print(f"  FN={fn}  TN={tn}")

Матрица ошибок сразу показывает, чего именно модель не умеет.

**Precision**, точность: из тех, кого модель назвала больными, сколько
действительно больны. Отвечает на вопрос «можно ли верить, когда модель
говорит да».

**Recall**, полнота: из всех больных сколько модель нашла. Отвечает на вопрос
«скольких мы пропустили».

Между ними всегда компромисс, и выбор зависит от цены ошибки. В медицинской
диагностике пропущенная опухоль страшнее ложной тревоги, поэтому здесь важнее
recall. В спам-фильтре наоборот: письмо от начальника в папке «спам» хуже,
чем реклама во входящих, поэтому важнее precision.

Метрику выбирает не алгоритм, а тот, кто понимает задачу.

## 10. Наивный Байес

Второй метод сегодняшнего занятия устроен принципиально иначе. kNN ничего
не вычисляет заранее и хранит всю выборку. Наивный Байес наоборот: он
на этапе обучения считает несколько чисел, а выборку выбрасывает.

### Откуда берется формула

В основе теорема Байеса. Она связывает то, что мы хотим знать, с тем,
что можно оценить по данным:

$$P(y = c \mid x) = \frac{P(x \mid y = c) \cdot P(y = c)}{P(x)}$$

Разберем по частям.

- $P(y = c \mid x)$ — **апостериорная** вероятность. Насколько вероятен класс
  $c$, если мы уже увидели объект $x$. Это ровно то, что нужно для ответа.
- $P(y = c)$ — **априорная** вероятность класса. Насколько класс вообще
  часто встречается, еще до того, как мы посмотрели на признаки.
  Оценивается просто долей класса в обучающей выборке.
- $P(x \mid y = c)$ — **правдоподобие**. Насколько типичен такой объект
  для этого класса.
- $P(x)$ — вероятность увидеть такой объект вообще

### Как из вероятностей получается ответ

Здесь стоит остановиться: это место обычно проскакивают, а оно ключевое.

Теорема Байеса считает вероятность **для каждого класса отдельно**.
Сортов три, значит получится три числа: $P(y = 0 \mid x)$,
$P(y = 1 \mid x)$ и $P(y = 2 \mid x)$. Но модель должна выдать **один**
ответ, а не три числа. Нужно правило выбора.

Правило самое естественное: назвать тот класс, чья вероятность больше.
Записывается это через $\arg\max$ — не само значение максимума, а тот
аргумент, на котором максимум достигается, то есть номер класса:

$$a(x) = \arg\max_{c} \; P(y = c \mid x)$$

Вот здесь функция $a(x)$ из второго раздела получает конкретный вид.
Подставим в нее теорему Байеса:

$$a(x) = \arg\max_{c} \; \frac{P(x \mid y = c) \, P(y = c)}{P(x)}$$

Осталась деталь, которая экономит всю работу. У всех трех дробей
знаменатель одинаковый: $P(x)$ от класса не зависит. Деление на одно
и то же положительное число не меняет порядок чисел — кто был больше,
тот больше и остался. Значит для выбора максимума знаменатель можно
выбросить:

$$a(x) = \arg\max_{c} \; P(y = c) \cdot P(x \mid y = c)$$

Это важно, потому что именно $P(x)$ было бы труднее всего оценить: это
распределение самих данных, многомерное и никак нам не заданное. Мы
не научились его считать — мы обошли необходимость его считать.

### В чем наивность

Осталась одна проблема, и она серьезная. Правдоподобие $P(x \mid y = c)$ —
это совместное распределение всех тринадцати признаков сразу. Чтобы оценить
его по выборке честно, данных нужно экспоненциально много: комбинаций
значений больше, чем объектов в любом реальном датасете.

Метод решает эту проблему грубо. Он предполагает, что при известном классе
признаки **независимы**, и тогда совместная вероятность распадается
в произведение одномерных:

$$P(x \mid y = c) = \prod_{j=1}^{d} P(x_j \mid y = c)$$

Вот это предположение и называется наивным. Оно почти всегда неверно:
в вине содержание флавоноидов и общее содержание фенолов очевидно связаны.
Метод все равно работает, и это одна из самых приятных странностей
машинного обучения. Причина в том, что для выбора класса не нужны точные
вероятности, достаточно правильного порядка.

Итоговое решающее правило:

$$a(x) = \arg\max_{c} \; P(y = c) \prod_{j=1}^{d} P(x_j \mid y = c)$$

### Гауссовский вариант

Остается сказать, как оценивать одномерные $P(x_j \mid y = c)$ для числовых
признаков. Самое простое предположение: внутри каждого класса признак
распределен нормально.

$$P(x_j \mid y = c) = \frac{1}{\sqrt{2\pi\sigma_{cj}^2}}
  \exp\left(-\frac{(x_j - \mu_{cj})^2}{2\sigma_{cj}^2}\right)$$

Тогда все обучение сводится к тому, чтобы посчитать по обучающей выборке
три набора чисел:

$$\hat{P}(y = c) = \frac{n_c}{n}, \qquad
  \hat{\mu}_{cj} = \frac{1}{n_c}\sum_{i:\, y_i = c} x_{ij}, \qquad
  \hat{\sigma}^2_{cj} = \frac{1}{n_c}\sum_{i:\, y_i = c} (x_{ij} - \hat{\mu}_{cj})^2$$

Это доля класса, среднее и дисперсия каждого признака внутри класса.
Для трех классов и тринадцати признаков получается семьдесят восемь чисел
плюс три доли. Вся модель.

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb = GaussianNB()
nb.fit(X_train, y_train)

print(f"наивный Байес:      {accuracy_score(y_test, nb.predict(X_test)):.3f}")
print(f"kNN с масштабом:    {acc_scaled:.3f}")
print(f"kNN без масштаба:   {acc_raw:.3f}")
print(f"baseline:           {baseline_acc:.3f}")
print()
print("вся выученная модель:")
print("  априорные вероятности классов:", nb.class_prior_.round(3))
print("  таблица средних:              ", nb.theta_.shape)
print("  таблица дисперсий:            ", nb.var_.shape)

Три вещи, которые стоит обсудить.

Первая: наивный Байес обыграл kNN. Метод, который считает несколько средних
и дисперсий, оказался лучше метода, который хранит всю выборку. Более сложное
не значит более точное.

Вторая: kNN без масштабирования проиграл всем. Мы это уже разобрали.

Третья, и она важнее двух первых. Наивный Байес показал единицу, то есть
не ошибся ни разу. Правильная реакция на такое число — не радость,
а настороженность. В тестовой выборке пятьдесят четыре объекта. Одна ошибка
опустила бы результат до 0.98, и разница между 1.00 и 0.98 здесь ничего
не значит: она в пределах случайности деления на две части.

Запомните это ощущение. Когда метрика подозрительно хороша, первым делом
надо искать не повод для гордости, а ошибку в постановке эксперимента.

Наивный Байес работает без масштабирования, и это не случайность. Он не
считает расстояний: для каждого признака отдельно оценивается свое
распределение внутри каждого класса. Единицы измерения при этом не важны.

Посмотрим, что именно он выучил.

In [ ]:
feature_idx = wine.feature_names.index("proline")

fig, ax = plt.subplots(figsize=(9, 4))
grid = np.linspace(X_train[:, feature_idx].min(), X_train[:, feature_idx].max(), 300)
colors = ["#0072CE", "#0F1418", "#6B7280"]

for c in range(3):
    mean = nb.theta_[c, feature_idx]
    std = np.sqrt(nb.var_[c, feature_idx])
    density = np.exp(-0.5 * ((grid - mean) / std) ** 2) / (std * np.sqrt(2 * np.pi))
    ax.plot(grid, density, color=colors[c], lw=2, label=f"класс {c}")
    ax.hist(X_train[y_train == c, feature_idx], bins=15, density=True,
            alpha=0.2, color=colors[c])

ax.set_xlabel("proline")
ax.set_ylabel("плотность")
ax.set_title("Что выучил наивный Байес: по нормальному распределению на класс")
ax.legend()
plt.tight_layout()
plt.show()

Гистограммы это реальные данные, кривые это то, чем метод их заменил.
Видно, что приближение грубое, и видно, что для разделения классов
его достаточно.

### Наивный Байес на тексте

Классическое применение метода — фильтрация спама. Здесь признаки другие:
не числа, а количества слов. Поэтому и распределение берут другое,
мультиномиальное:

$$P(w \mid c) = \frac{N_{wc} + \alpha}{N_c + \alpha \, |V|}$$

где $N_{wc}$ — сколько раз слово $w$ встретилось в текстах класса $c$,
$N_c$ — общее число слов в классе, $|V|$ — размер словаря.

Слагаемое $\alpha$ здесь не украшение. Без него слово, ни разу не встреченное
в спаме, дает $P(w \mid \text{спам}) = 0$, а ноль в произведении обнуляет
все выражение целиком: одно незнакомое слово перечеркивает все остальные.
Такое сглаживание называют аддитивным, или сглаживанием Лапласа при $\alpha = 1$.

Предположение о независимости здесь означает «слова в сообщении появляются
независимо друг от друга». Это очевидно неправда, и это все равно работает.

In [ ]:
spam = [
    "выиграйте приз прямо сейчас",
    "бесплатный кредит без проверок",
    "срочно подтвердите карту иначе блокировка",
    "заработок от пятидесяти тысяч в день",
    "вы выиграли миллион перейдите по ссылке",
    "только сегодня скидка девяносто процентов",
    "кредит без справок и поручителей срочно",
    "получите бесплатный приз перейдите по ссылке",
]
ham = [
    "во сколько завтра встречаемся",
    "отправил тебе отчет посмотри пожалуйста",
    "не забудь про семинар в субботу",
    "давай перенесем созвон на час позже",
    "спасибо за помощь с задачей",
    "я задержусь минут на пятнадцать",
    "посмотри пожалуйста мой код там ошибка",
    "встреча перенесена на завтра утром",
]

texts = spam + ham
labels = np.array([1] * len(spam) + [0] * len(ham))
print(f"{len(texts)} сообщений, из них спама {labels.sum()}")

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

probe = [
    "срочно получите бесплатный приз",
    "давай перенесем встречу на завтра",
    "подтвердите карту иначе блокировка",
]

# TODO: превратить тексты в матрицу счетчиков слов (vectorizer, X_text),
# обучить MultinomialNB (nb_text) и получить вероятности спама
# для сообщений из probe в массив probe_proba
raise NotImplementedError

In [ ]:
# --- проверка ---

# 1. строк в матрице столько же, сколько сообщений
assert X_text.shape[0] == len(texts), "число строк не равно числу сообщений"

# 2. колонок столько же, сколько слов в словаре
assert X_text.shape[1] == len(vectorizer.vocabulary_), "число колонок не равно размеру словаря"

# 3. в матрице только неотрицательные целые: это счетчики слов
assert (X_text.toarray() >= 0).all(), "в матрице счетчиков появилось отрицательное число"

# 4. вероятности это вероятности
assert ((probe_proba >= 0) & (probe_proba <= 1)).all(), "вероятность вне отрезка [0, 1]"

# 5. содержательная проверка: первое и третье сообщения спамные, второе нет
assert probe_proba[0] > 0.5, "явный спам не распознан"
assert probe_proba[1] < 0.5, "обычное сообщение принято за спам"
assert probe_proba[2] > 0.5, "явный спам не распознан"

print("проверки пройдены, вероятности спама:", probe_proba.round(3))

`CountVectorizer` превращает тексты в таблицу: строка это сообщение, колонка
это слово из словаря, в клетке количество вхождений. Такое представление
называется мешком слов, потому что порядок слов теряется полностью.

Посмотрим, какие слова метод считает самыми спамными. Для этого сравним
логарифмы вероятностей слова в двух классах: чем больше разность,
тем сильнее слово тянет в сторону спама.

In [ ]:
log_ratio = nb_text.feature_log_prob_[1] - nb_text.feature_log_prob_[0]
words = np.array(vectorizer.get_feature_names_out())
order = np.argsort(log_ratio)

print("самые спамные слова:")
for w in words[order[-8:]][::-1]:
    print("   ", w)
print()
print("самые нормальные слова:")
for w in words[order[:8]]:
    print("   ", w)

## 11. Что запомнить

**Про механику.** Любая модель в scikit-learn это `fit` и `predict`.
Данные делятся на обучающую и тестовую часть до всего остального.
Преобразования настраиваются на трейне и применяются к тесту, а не наоборот.

**Про kNN.** Обучения нет, вся работа при предсказании. Чувствителен
к масштабу признаков до полной потери качества. Единственный параметр `k`
управляет тем, насколько модель подстраивается под данные.

**Про наивный Байес.** Считает несколько чисел и выбрасывает выборку.
Не зависит от масштаба. Предположение о независимости признаков почти всегда
неверно, а метод все равно работает.

**Про метрики.** Accuracy сама по себе не значит ничего, пока не известно
распределение классов. Матрица ошибок показывает, какие именно ошибки
делает модель. Precision и recall отвечают на разные вопросы, и выбор между
ними определяется ценой ошибки в конкретной задаче.

**Главное.** Baseline считается первым. Модель, которая не обыгрывает
константный ответ, не имеет права на существование.

## Домашнее задание к следующему занятию

1. Посмотреть лекции 2 и 3 из «Тренировок по ML», ссылки в `docs/lectures.md`
2. Завести аккаунт на Hugging Face, он понадобится со второго занятия
3. Доделать лабораторную, если не успели на паре